In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform


def ripley_k_translation(
    points,
    r_values,
    autocor,
    x0,
    y0,
):
    """
    Estimate Ripley's K-function using translation edge correction.

    Parameters
    ----------
    points : array-like, shape (n, 2)
        Point coordinates inside the observation window.
        Column 0 is x; column 1 is y.

    r_values : array-like
        Radii at which K(r) is evaluated.

    autocor : ndarray, shape (2H-1, 2W-1)
        Discrete overlap-area lookup table generated from the
        binary observation-window mask.

    x0, y0 : int
        Autocorrelation indices corresponding to zero displacement.

    Returns
    -------
    K : ndarray
        Translation edge-corrected K-function.
    """

    points = np.asarray(points, dtype=float)
    r_values = np.asarray(r_values, dtype=float)
    autocor = np.asarray(autocor, dtype=float)

    # Validate point array
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("points must have shape (n, 2).")

    n = len(points)

    if n < 2:
        raise ValueError("At least two points are required.")

    if not np.all(np.isfinite(points)):
        raise ValueError("points contains NaN or infinite values.")

    if not np.all(np.isfinite(r_values)):
        raise ValueError("r_values contains NaN or infinite values.")

    if np.any(r_values < 0):
        raise ValueError("r_values must be non-negative.")

    if autocor.ndim != 2:
        raise ValueError("autocor must be a two-dimensional array.")

    x0 = int(x0)
    y0 = int(y0)

    if not (
        0 <= y0 < autocor.shape[0]
        and 0 <= x0 < autocor.shape[1]
    ):
        raise ValueError("The zero-displacement index is outside autocor.")

    # Area of the observation window at zero displacement
    area = autocor[y0, x0]

    if not np.isfinite(area) or area <= 0:
        raise ValueError(
            "autocor[y0, x0] must be a positive finite window area."
        )

    # Pairwise distances
    distances = squareform(pdist(points))

    # Exclude self-pairs
    np.fill_diagonal(distances, np.inf)

    # Pairwise displacement vectors
    dx = points[:, 0][:, None] - points[:, 0][None, :]
    dy = points[:, 1][:, None] - points[:, 1][None, :]

    # Map displacements to the autocorrelation grid
    x_indices = np.rint(x0 + dx).astype(int)
    y_indices = np.rint(y0 + dy).astype(int)

    valid_overlap_indices = (
        (0 <= x_indices)
        & (x_indices < autocor.shape[1])
        & (0 <= y_indices)
        & (y_indices < autocor.shape[0])
    )

    overlap_areas = np.zeros_like(distances, dtype=float)

    overlap_areas[valid_overlap_indices] = autocor[
        y_indices[valid_overlap_indices],
        x_indices[valid_overlap_indices],
    ]

    valid_pairs = (
        valid_overlap_indices
        & np.isfinite(overlap_areas)
        & (overlap_areas > 0)
        & np.isfinite(distances)
    )

    # Translation weights: |W| / overlap_area
    weights = np.zeros_like(distances, dtype=float)
    weights[valid_pairs] = area / overlap_areas[valid_pairs]

    # Ordered-pair estimator
    K = np.empty_like(r_values, dtype=float)

    denominator = n * (n - 1)

    for k, r in enumerate(r_values):
        included_pairs = valid_pairs & (distances <= r)
        total = weights[included_pairs].sum()

        K[k] = area * total / denominator

    return K

### Test data

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely.affinity import translate
from scipy.spatial.distance import pdist, squareform
from skimage.draw import polygon
from scipy.signal import fftconvolve

# create a fake set of points
points = np.random.uniform(0,100,2000).reshape(-1,2)
filt = points[:,0] + points[:,1] > 100
points = points[filt]

hull = ConvexHull(points)
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-o')

window = []
for vertix in hull.vertices:
    window.append([points[vertix, 1], points[vertix, 0]])
window = np.array(window)

print(window.shape)
# to see the polygon just run
#Polygon(window)

window = Polygon(window)

In [ ]:
# create binary mask
x, y = window.exterior.coords.xy
x = np.array(x)
y = np.array(y)

height, width = int(np.max(y) + 1), int(np.max(x) + 1)
mask = np.zeros((height, width), dtype=np.uint8)

rr, cc = polygon(y, x, shape=mask.shape)
mask[rr, cc] = 1
plt.imshow(mask)
plt.figure()
autocor = fftconvolve(mask, mask[::-1, ::-1], mode='full')
y0, x0 = np.unravel_index(np.argmax(autocor), autocor.shape)
plt.imshow(autocor)
print(y0, x0)

In [ ]:
r_values = np.linspace(0.0, 60, 40)

K = ripley_k_translation(points, r_values, autocor, x0, y0)
K_csr = np.pi * r_values**2

plt.figure()
plt.plot(r_values, K, label="Ripley's K (translation)")
plt.plot(r_values, K_csr, "--", label="CSR (πr²)")
plt.xlabel("r")
plt.ylabel("K(r)")
plt.legend()
plt.title("Ripley's K-function (Arbitrary Polygon Window)")
plt.show()

In [ ]:
plt.figure()
plt.plot(r_values, (K - K_csr)/K.std(), label="Ripley's K (edge-corrected) - CRS")
plt.plot(r_values, r_values * 0, label="zero level, above: clustering, below: dispersion")

plt.xlabel("r")
plt.ylabel("K(r)")
plt.legend()
plt.title("Difference between estimated Ripley's K-function and CSR")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def generate_and_plot_datasets(seed=41, show_plot=True):
    rng = np.random.default_rng(seed)

    # Observation window
    plane_size = 100
    buffer = 100

    # Coordinates of the central observation window
    window_min = buffer
    window_max = buffer + plane_size

    # Full region used to generate surrounding points
    simulation_size = plane_size + 2 * buffer
    simulation_area = simulation_size**2

    # Cluster-process parameters
    base_num_centres = 20
    points_levels = [20, 50, 100]
    std_levels = [2, 5, 10]

    # Preserve the same centre density in the larger generating region
    area_multiplier = (simulation_size / plane_size) ** 2
    scaled_num_centres = int(base_num_centres * area_multiplier)

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(12, 12),
        sharex=True,
        sharey=True,
    )

    datasets = {}

    for row, std_dev in enumerate(std_levels):
        for col, points_per_centre in enumerate(points_levels):

            # Generate centres throughout the full surrounding region
            centres = rng.uniform(
                0,
                simulation_size,
                size=(scaled_num_centres, 2),
            )

            clusters = []

            for cx, cy in centres:
                cluster = np.column_stack([
                    rng.normal(
                        loc=cx,
                        scale=std_dev,
                        size=points_per_centre,
                    ),
                    rng.normal(
                        loc=cy,
                        scale=std_dev,
                        size=points_per_centre,
                    ),
                ])

                clusters.append(cluster)

            all_points = np.vstack(clusters)

            # Select points in the central observation window
            inside_window = (
                (all_points[:, 0] >= window_min)
                & (all_points[:, 0] < window_max)
                & (all_points[:, 1] >= window_min)
                & (all_points[:, 1] < window_max)
            )

            window_points = all_points[inside_window].copy()

            # Shift focal points into local 0–100 window coordinates
            window_points -= np.array([window_min, window_min])

            # Known nominal intensity of the simulated point process
            nominal_intensity = (
                scaled_num_centres * points_per_centre
                / simulation_area
            )

            dataset_id = (row, col)

            datasets[dataset_id] = {
                "all_points": all_points,
                "window_points": window_points,
                "starting_points_mask": inside_window.astype(bool),
                "centres": centres,
                "std_dev": std_dev,
                "points_per_centre": points_per_centre,
                "plane_size": plane_size,
                "buffer": buffer,
                "simulation_size": simulation_size,
                "simulation_area": simulation_area,
                "nominal_intensity": nominal_intensity,
                "seed": seed,
            }

            # Plot focal points in local window coordinates
            ax = axes[row, col]

            ax.scatter(
                window_points[:, 0],
                window_points[:, 1],
                s=8,
                alpha=0.6,
            )

            ax.set_xlim(0, plane_size)
            ax.set_ylim(0, plane_size)
            ax.set_aspect("equal")
            ax.set_title(
                f"$\\sigma$={std_dev}, "
                f"m={points_per_centre}, "
                f"N={len(window_points)}"
            )

    for ax in axes[-1, :]:
        ax.set_xlabel("X")

    for ax in axes[:, 0]:
        ax.set_ylabel("Y")

    fig.suptitle(
        f"Buffered synthetic datasets "
        f"(seed {seed})",
        y=0.99,
        fontsize=14,
    )

    fig.tight_layout()

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return datasets

In [ ]:
seeds_to_test = [101, 202, 303]

all_runs_data = {}

for seed in seeds_to_test:
    print(f"Generating data for seed {seed}...")

    all_runs_data[seed] = generate_and_plot_datasets(
        seed=seed,
        show_plot=True,
    )

In [ ]:
import numpy as np
from scipy.signal import fftconvolve


def create_fixed_square_window(plane_size=100):
    mask = np.ones(
        (plane_size, plane_size),
        dtype=np.uint8,
    )

    autocor = fftconvolve(
        mask,
        mask[::-1, ::-1],
        mode="full",
    )

    y0, x0 = np.unravel_index(
        np.argmax(autocor),
        autocor.shape,
    )

    area = autocor[y0, x0]

    if not np.isclose(area, mask.sum()):
        raise ValueError(
            "autocor[y0, x0] does not equal the mask area."
        )

    return {
        "mask": mask,
        "autocor": autocor,
        "x0": x0,
        "y0": y0,
        "area": float(area),
        "plane_size": plane_size,
    }


# This line actually creates the object
window_info = create_fixed_square_window(plane_size=100)

print("window_info created")
print("Mask shape:", window_info["mask"].shape)
print("Autocorrelation shape:", window_info["autocor"].shape)
print("Zero-displacement index:", window_info["x0"], window_info["y0"])
print("Window area:", window_info["area"])

In [ ]:
import numpy as np

# Evaluation parameters
r_max = 25.0
num_steps = 50

# Avoid duplicate zero-distance handling if desired.
# Including zero is mathematically valid and should normally give K(0)=0.
r_values = np.linspace(0.0, r_max, num_steps)

# Fixed translation-correction information
autocor = window_info["autocor"]
x0 = window_info["x0"]
y0 = window_info["y0"]

# Results for all seeds and configurations
all_seeds_ripley_results = {}

# Optional metadata for quality control
all_seeds_ripley_metadata = {}

for seed, datasets in all_runs_data.items():
    print(f"Calculating Ripley's K for seed {seed}...")

    seed_results = {}
    seed_metadata = {}

    for dataset_id, dataset in datasets.items():
        points = np.asarray(
            dataset["window_points"],
            dtype=float,
        )

        n_points = len(points)

        print(
            f"  Processing dataset {dataset_id}: "
            f"n={n_points}"
        )

        if n_points < 2:
            print(
                f"    Skipping dataset {dataset_id}: "
                "fewer than two points."
            )

            seed_results[dataset_id] = np.full(
                r_values.shape,
                np.nan,
                dtype=float,
            )

            seed_metadata[dataset_id] = {
                "n_points": n_points,
                "valid": False,
            }

            continue

        K_values = ripley_k_translation(
            points=points,
            r_values=r_values,
            autocor=autocor,
            x0=x0,
            y0=y0,
        )

        seed_results[dataset_id] = K_values

        seed_metadata[dataset_id] = {
            "n_points": n_points,
            "valid": True,
            "std_dev": dataset["std_dev"],
            "points_per_centre": dataset["points_per_centre"],
            "nominal_intensity": dataset["nominal_intensity"],
        }

    all_seeds_ripley_results[seed] = seed_results
    all_seeds_ripley_metadata[seed] = seed_metadata

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
})

fig, axes = plt.subplots(
    3,
    3,
    figsize=(14, 11),
    sharex=True,
    sharey=True,
)

fig.suptitle(
    "Translation-corrected Ripley's K-function across seeds",
    fontsize=14,
    weight="bold",
    y=0.98,
)

K_csr = np.pi * r_values**2

seeds = list(all_seeds_ripley_results.keys())
colors = plt.cm.plasma(
    np.linspace(0.1, 0.85, len(seeds))
)

for row in range(3):
    for col in range(3):

        dataset_id = (row, col)
        ax = axes[row, col]

        curves = []
        plotted_seed_labels = []

        for i, seed in enumerate(seeds):

            if dataset_id not in all_seeds_ripley_results[seed]:
                continue

            K = np.asarray(
                all_seeds_ripley_results[seed][dataset_id],
                dtype=float,
            )

            if K.shape != r_values.shape:
                raise ValueError(
                    f"Shape mismatch for seed {seed}, "
                    f"dataset {dataset_id}: "
                    f"K has shape {K.shape}, "
                    f"r_values has shape {r_values.shape}."
                )

            if np.all(np.isnan(K)):
                continue

            ax.plot(
                r_values,
                K,
                color=colors[i],
                linewidth=1.3,
                alpha=0.65,
                label=f"Seed {seed}",
            )

            curves.append(K)
            plotted_seed_labels.append(f"Seed {seed}")

        # Plot CSR only as a reference
        ax.plot(
            r_values,
            K_csr,
            color="#d62728",
            linestyle="--",
            linewidth=1.3,
            label="CSR reference",
        )

        # Mean and standard-deviation band across seeds
        if len(curves) >= 2:
            curves = np.asarray(curves)

            mean_K = np.nanmean(curves, axis=0)
            std_K = np.nanstd(curves, axis=0, ddof=1)

            ax.plot(
                r_values,
                mean_K,
                color="black",
                linewidth=2.2,
                label="Mean across seeds",
            )

            ax.fill_between(
                r_values,
                mean_K - std_K,
                mean_K + std_K,
                color="black",
                alpha=0.12,
                label="±1 SD",
            )

        # Use simulation parameters in the title if metadata exist
        title = f"Dataset ({row}, {col})"

        if (
            seed in all_seeds_ripley_metadata
            and dataset_id in all_seeds_ripley_metadata[seed]
        ):
            metadata = all_seeds_ripley_metadata[
                seed
            ][dataset_id]

            if "std_dev" in metadata:
                title += (
                    f"\nσ={metadata['std_dev']}, "
                    f"m={metadata['points_per_centre']}"
                )

        ax.set_title(title, pad=8)

        if row == 2:
            ax.set_xlabel("Distance, r")

        if col == 0:
            ax.set_ylabel("K(r)")

        ax.grid(True, alpha=0.3)

# Create a deduplicated figure-level legend
handles, labels = axes[0, 0].get_legend_handles_labels()

unique_handles = {}
for handle, label in zip(handles, labels):
    unique_handles[label] = handle

fig.legend(
    unique_handles.values(),
    unique_handles.keys(),
    loc="upper left",
    bbox_to_anchor=(1.02, 0.95),
    frameon=True,
    facecolor="white",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
})

fig, axes = plt.subplots(
    3,
    3,
    figsize=(10, 8),
    sharex=True,
    sharey=True,
)

fig.suptitle(
    r"Translation-corrected $L(r)-r$ across seeds",
    fontsize=14,
    weight="bold",
    y=0.98,
)

seeds = list(all_seeds_ripley_results.keys())
colors = plt.cm.plasma(
    np.linspace(0.1, 0.85, len(seeds))
)

for row in range(3):
    for col in range(3):

        dataset_id = (row, col)
        ax = axes[row, col]

        curves = []

        for i, seed in enumerate(seeds):

            if dataset_id not in all_seeds_ripley_results[seed]:
                continue

            K = np.asarray(
                all_seeds_ripley_results[seed][dataset_id],
                dtype=float,
            )

            if K.shape != r_values.shape:
                raise ValueError(
                    f"Shape mismatch for seed {seed}, "
                    f"dataset {dataset_id}: "
                    f"K shape={K.shape}, "
                    f"r_values shape={r_values.shape}"
                )

            if np.all(np.isnan(K)):
                continue

            if np.any(K < 0):
                raise ValueError(
                    f"Negative K-values for seed {seed}, "
                    f"dataset {dataset_id}."
                )

            # L(r) - r transformation
            L_minus_r = np.sqrt(K / np.pi) - r_values

            ax.plot(
                r_values,
                L_minus_r,
                color=colors[i],
                linewidth=1.4,
                alpha=0.65,
                label=f"Seed {seed}",
            )

            curves.append(L_minus_r)

        # Zero corresponds to the CSR expectation
        ax.axhline(
            0.0,
            color="#d62728",
            linestyle="--",
            linewidth=1.4,
            label="CSR reference: L(r)-r=0",
        )

        # Mean and ±1 SD across seeds
        if len(curves) >= 2:
            curves = np.asarray(curves)

            mean_curve = np.nanmean(curves, axis=0)
            std_curve = np.nanstd(
                curves,
                axis=0,
                ddof=1,
            )

            ax.plot(
                r_values,
                mean_curve,
                color="black",
                linewidth=2.2,
                label="Mean across seeds",
            )

            ax.fill_between(
                r_values,
                mean_curve - std_curve,
                mean_curve + std_curve,
                color="black",
                alpha=0.12,
                label="±1 SD",
            )

        ax.set_title(
            f"Dataset ({row}, {col})",
            pad=8,
        )

        if row == 2:
            ax.set_xlabel("Distance, r")

        if col == 0:
            ax.set_ylabel(r"$L(r)-r$")

        ax.grid(True, alpha=0.3)

# Deduplicate the figure-level legend
handles, labels = axes[0, 0].get_legend_handles_labels()

by_label = dict(zip(labels, handles))

fig.legend(
    by_label.values(),
    by_label.keys(),
    loc="upper left",
    bbox_to_anchor=(1.02, 0.95),
    frameon=True,
    facecolor="white",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
)

fig.savefig(
    "Figures/translation_corrected_L_minus_r.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

### Simple Ripley for comparison

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, cdist, squareform


def ripley_k(
    starting_points,
    r_values,
    area=None,
    all_points=None,
    intensity=None,
    starting_indices=None,
):
    """
    Estimate Ripley's K-function.

    Two modes are supported:

    1. Standard uncorrected estimator:
       all_points=None

    2. Buffered estimator:
       starting_points are focal points and all_points contains
       the surrounding point process.

    Parameters
    ----------
    starting_points : ndarray, shape (n_start, 2)
        Focal points.

    r_values : ndarray
        Radii at which K(r) is evaluated.

    area : float, optional
        Observation-window area. Required for the standard
        single-window estimator.

    all_points : ndarray, shape (n_all, 2), optional
        Full buffered point pattern. Required for buffered mode.

    intensity : float, optional
        Intensity of the process generating all_points. Required
        for buffered mode.

    starting_indices : ndarray, optional
        Column indices in all_points corresponding to starting_points.
        This is needed to remove self-pairs in buffered mode.
    """

    starting_points = np.asarray(
        starting_points,
        dtype=float,
    )

    r_values = np.asarray(
        r_values,
        dtype=float,
    )

    if (
        starting_points.ndim != 2
        or starting_points.shape[1] != 2
    ):
        raise ValueError(
            "starting_points must have shape (n, 2)."
        )

    if len(starting_points) < 2:
        raise ValueError(
            "At least two starting points are required."
        )

    if np.any(r_values < 0):
        raise ValueError(
            "r_values must be non-negative."
        )

    n_start = len(starting_points)

    # ---------------------------------------------------------
    # Mode 1: standard uncorrected within-window estimator
    # ---------------------------------------------------------
    if all_points is None:

        if area is None or area <= 0:
            raise ValueError(
                "A positive area is required when all_points is None."
            )

        distances = squareform(
            pdist(starting_points)
        )

        np.fill_diagonal(
            distances,
            np.inf,
        )

        K = np.zeros_like(
            r_values,
            dtype=float,
        )

        denominator = n_start * (n_start - 1)

        for k, radius in enumerate(r_values):
            pair_count = np.sum(
                distances <= radius
            )

            K[k] = (
                area
                * pair_count
                / denominator
            )

        return K

    # ---------------------------------------------------------
    # Mode 2: buffered estimator
    # ---------------------------------------------------------
    all_points = np.asarray(
        all_points,
        dtype=float,
    )

    if (
        all_points.ndim != 2
        or all_points.shape[1] != 2
    ):
        raise ValueError(
            "all_points must have shape (n, 2)."
        )

    if intensity is None or intensity <= 0:
        raise ValueError(
            "A positive intensity is required in buffered mode."
        )

    if starting_indices is None:
        raise ValueError(
            "starting_indices are required to remove self-pairs "
            "in buffered mode."
        )

    starting_indices = np.asarray(
        starting_indices,
        dtype=int,
    )

    if len(starting_indices) != n_start:
        raise ValueError(
            "starting_indices must have one entry per starting point."
        )

    if np.any(
        (starting_indices < 0)
        | (starting_indices >= len(all_points))
    ):
        raise ValueError(
            "starting_indices contain invalid all_points indices."
        )

    distances = cdist(
        starting_points,
        all_points,
    )

    # Remove the focal point itself from each row
    distances[
        np.arange(n_start),
        starting_indices,
    ] = np.inf

    K = np.zeros_like(
        r_values,
        dtype=float,
    )

    denominator = n_start * intensity

    for k, radius in enumerate(r_values):
        pair_count = np.sum(
            distances <= radius
        )

        K[k] = pair_count / denominator

    return K

In [ ]:
import numpy as np

# Use exactly the same radii as for the translation estimator
r_values = np.linspace(
    0.0,
    25.0,
    50,
)

# Store buffered results separately
all_seeds_buffered_results = {}

for seed, datasets in all_runs_data.items():
    print(
        f"Calculating buffered Ripley's K "
        f"for seed {seed}..."
    )

    seed_results = {}

    for dataset_id, dataset in datasets.items():

        all_points = np.asarray(
            dataset["all_points"],
            dtype=float,
        )

        # This must be a Boolean array with one entry per all_points row
        if "starting_points_mask" not in dataset:
            raise KeyError(
                f"Dataset {dataset_id} for seed {seed} "
                "does not contain 'starting_points_mask'."
            )

        starting_points_mask = np.asarray(
            dataset["starting_points_mask"],
            dtype=bool,
        )

        if len(starting_points_mask) != len(all_points):
            raise ValueError(
                f"Mask length does not match all_points length "
                f"for seed {seed}, dataset {dataset_id}."
            )

        # Focal points remain in the same global coordinate system
        # as all_points
        starting_points = all_points[starting_points_mask]

        # Original column indices of the focal points in all_points
        starting_indices = np.flatnonzero(
            starting_points_mask
        )

        nominal_intensity = float(
            dataset["nominal_intensity"]
        )

        print(
            f"  Processing dataset {dataset_id}: "
            f"n_start={len(starting_points)}, "
            f"n_all={len(all_points)}, "
            f"lambda={nominal_intensity:.5f}"
        )

        if len(starting_points) < 2:
            seed_results[dataset_id] = np.full(
                r_values.shape,
                np.nan,
                dtype=float,
            )
            continue

        K_buffered = ripley_k(
            starting_points=starting_points,
            r_values=r_values,
            all_points=all_points,
            intensity=nominal_intensity,
            starting_indices=starting_indices,
        )

        seed_results[dataset_id] = K_buffered

    all_seeds_buffered_results[seed] = seed_results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
})

fig, axes = plt.subplots(
    3,
    3,
    figsize=(14, 11),
    sharex=True,
    sharey=True,
)

fig.suptitle(
    "Buffered Ripley's K-function across seeds",
    fontsize=14,
    weight="bold",
    y=0.98,
)

# Use the same shared radii as the translation estimator
K_csr = np.pi * r_values**2

seeds = list(all_seeds_buffered_results.keys())

colors = plt.cm.plasma(
    np.linspace(0.1, 0.85, len(seeds))
)

for row in range(3):
    for col in range(3):

        dataset_id = (row, col)
        ax = axes[row, col]

        curves = []

        for i, seed in enumerate(seeds):

            if dataset_id not in all_seeds_buffered_results[seed]:
                continue

            K = np.asarray(
                all_seeds_buffered_results[seed][dataset_id],
                dtype=float,
            )

            if K.shape != r_values.shape:
                raise ValueError(
                    f"Shape mismatch for seed {seed}, "
                    f"dataset {dataset_id}: "
                    f"K shape={K.shape}, "
                    f"r_values shape={r_values.shape}"
                )

            if np.all(np.isnan(K)):
                continue

            ax.plot(
                r_values,
                K,
                color=colors[i],
                linewidth=1.3,
                alpha=0.65,
                label=f"Seed {seed}",
            )

            curves.append(K)

        # CSR is shown only as a reference.
        # The simulated datasets are clustered, so their expected
        # K-function is not generally equal to this line.
        ax.plot(
            r_values,
            K_csr,
            color="#d62728",
            linestyle="--",
            linewidth=1.3,
            label="CSR reference",
        )

        # Mean and ±1 SD across seeds
        if len(curves) >= 2:
            curves = np.asarray(curves)

            mean_K = np.nanmean(
                curves,
                axis=0,
            )

            std_K = np.nanstd(
                curves,
                axis=0,
                ddof=1,
            )

            ax.plot(
                r_values,
                mean_K,
                color="black",
                linewidth=2.2,
                label="Mean across seeds",
            )

            ax.fill_between(
                r_values,
                mean_K - std_K,
                mean_K + std_K,
                color="black",
                alpha=0.12,
                label="±1 SD",
            )

        # Add simulation parameters to the title when available
        title = f"Dataset ({row}, {col})"

        if (
            "all_seeds_ripley_metadata" in globals()
            and seed in all_seeds_ripley_metadata
            and dataset_id in all_seeds_ripley_metadata[seed]
        ):
            metadata = all_seeds_ripley_metadata[seed][dataset_id]

            if "std_dev" in metadata:
                title += (
                    f"\nσ={metadata['std_dev']}, "
                    f"m={metadata['points_per_centre']}"
                )

        ax.set_title(title, pad=8)

        if row == 2:
            ax.set_xlabel("Distance, r")

        if col == 0:
            ax.set_ylabel("K(r)")

        ax.grid(True, alpha=0.3)

# Deduplicate the global legend
handles, labels = axes[0, 0].get_legend_handles_labels()

by_label = dict(zip(labels, handles))

fig.legend(
    by_label.values(),
    by_label.keys(),
    loc="upper left",
    bbox_to_anchor=(1.02, 0.95),
    frameon=True,
    facecolor="white",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
)

plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
})

fig, axes = plt.subplots(
    3,
    3,
    figsize=(10, 8),
    sharex=True,
    sharey=True,
)

fig.suptitle(
    r"Buffered $L(r)-r$ across seeds",
    fontsize=14,
    weight="bold",
    y=0.98,
)

seeds = list(all_seeds_buffered_results.keys())

colors = plt.cm.plasma(
    np.linspace(0.1, 0.85, len(seeds))
)

for row in range(3):
    for col in range(3):

        dataset_id = (row, col)
        ax = axes[row, col]

        curves = []

        for i, seed in enumerate(seeds):

            if dataset_id not in all_seeds_buffered_results[seed]:
                continue

            K = np.asarray(
                all_seeds_buffered_results[seed][dataset_id],
                dtype=float,
            )

            if K.shape != r_values.shape:
                raise ValueError(
                    f"Shape mismatch for seed {seed}, "
                    f"dataset {dataset_id}: "
                    f"K shape={K.shape}, "
                    f"r_values shape={r_values.shape}"
                )

            if np.all(np.isnan(K)):
                continue

            finite_K = K[np.isfinite(K)]

            if np.any(finite_K < 0):
                raise ValueError(
                    f"Negative K-values for seed {seed}, "
                    f"dataset {dataset_id}."
                )

            # L(r) - r transformation
            L_minus_r = (
                np.sqrt(K / np.pi)
                - r_values
            )

            ax.plot(
                r_values,
                L_minus_r,
                color=colors[i],
                linewidth=1.4,
                alpha=0.65,
                label=f"Seed {seed}",
            )

            curves.append(L_minus_r)

        # CSR reference: L(r) - r = 0
        ax.axhline(
            0.0,
            color="#d62728",
            linestyle="--",
            linewidth=1.4,
            label="CSR reference",
        )

        # Mean and ±1 SD across seeds
        if len(curves) >= 2:
            curves = np.asarray(curves)

            mean_curve = np.nanmean(
                curves,
                axis=0,
            )

            std_curve = np.nanstd(
                curves,
                axis=0,
                ddof=1,
            )

            ax.plot(
                r_values,
                mean_curve,
                color="black",
                linewidth=2.2,
                label="Mean across seeds",
            )

            ax.fill_between(
                r_values,
                mean_curve - std_curve,
                mean_curve + std_curve,
                color="black",
                alpha=0.12,
                label="±1 SD",
            )

        # Add simulation parameters if available
        title = f"Dataset ({row}, {col})"

        if (
            "all_seeds_ripley_metadata" in globals()
            and seed in all_seeds_ripley_metadata
            and dataset_id in all_seeds_ripley_metadata[seed]
        ):
            metadata = all_seeds_ripley_metadata[
                seed
            ][dataset_id]

            if "std_dev" in metadata:
                title += (
                    f"\nσ={metadata['std_dev']}, "
                    f"m={metadata['points_per_centre']}"
                )

        ax.set_title(title, pad=8)

        if row == 2:
            ax.set_xlabel("Distance, r")

        if col == 0:
            ax.set_ylabel(r"$L(r)-r$")

        ax.grid(True, alpha=0.3)

# Deduplicate the figure-level legend
handles, labels = axes[0, 0].get_legend_handles_labels()

by_label = dict(zip(labels, handles))

fig.legend(
    by_label.values(),
    by_label.keys(),
    loc="upper left",
    bbox_to_anchor=(1.02, 0.95),
    frameon=True,
    facecolor="white",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.93],
)

# Optional SVG export
os.makedirs("Figures", exist_ok=True)

fig.savefig(
    "Figures/buffered_L_minus_r.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()

### Analysis by dox/Tau status

In [ ]:
import csv
import pickle
from skimage import io

# 1. Define configurations for both datasets
dataset_configs = {
    1: {
        "image_path": "C1-Lysosome_masked_crops_1024.tif",
        "csv_path": "PhTau_histogram_clusters.csv",
        "coord_path": "processed_data/spot_coordinates.pkl",
        "delaunay_path": "processed_data/delaunay_results.pkl",
    },
    2: {
        "image_path": "C1-cell_gallery_1024x1024_6ch.tif",
        "csv_path": "gallery_phtau_ground_truth.csv",
        "coord_path": "processed_data_2/spot_coordinates.pkl",
        "delaunay_path": "processed_data_2/delaunay_results.pkl",
    },
}

# 2. Initialise master data containers
images = {}
n_images = {}
labels = {}
categories = {}
coord_map = {}
delaunay_objects_dict = {}

# 3. Process both datasets in a single loop
for idx, config in dataset_configs.items():
    # Load image and get shape
    images[idx] = io.imread(config["image_path"])
    n_images[idx] = images[idx].shape[0]

    # Parse CSV labels
    labels[idx] = []
    with open(config["csv_path"], "r") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for row in reader:
            val = row[-1]
            if val == "0":
                labels[idx].append(["0", "dox+/PhTau+"])
            elif val == "1":
                labels[idx].append(["1", "dox+/PhTau-"])

    # Pad untreated cells
    while len(labels[idx]) < n_images[idx]:
        labels[idx].append(["2", "dox-"])

    # Process categories
    cat_set = {tuple(item) for item in labels[idx]}
    sorted_pairs = sorted(cat_set, key=lambda x: int(x[0]), reverse=True)
    categories[idx] = [pair[1] for pair in sorted_pairs]

    # Load Pickle files
    with open(config["coord_path"], "rb") as f:
        coord_map[idx] = pickle.load(f)

    with open(config["delaunay_path"], "rb") as f:
        delaunay_objects_dict[idx] = pickle.load(f)

In [ ]:
import numpy as np
from scipy.ndimage import binary_fill_holes
from scipy.signal import fftconvolve


def create_2d_cell_mask(image):
    """
    Convert one image into a two-dimensional cell mask.

    For a multichannel image, a pixel is considered part of the
    cell if any channel is non-zero.
    """

    image = np.asarray(image)

    if image.ndim == 2:
        raw_mask = image > 0

    elif image.ndim == 3:
        # Expected shape: (H, W, C)
        raw_mask = np.any(image > 0, axis=-1)

    else:
        raise ValueError(
            f"Expected a 2D or 3D image, got shape {image.shape}."
        )

    cell_mask = binary_fill_holes(raw_mask)

    return cell_mask.astype(np.uint8)


def create_mask_autocorrelation(cell_mask):
    """
    Create the translation-overlap lookup table for a 2D mask.
    """

    if cell_mask.ndim != 2:
        raise ValueError(
            "cell_mask must be two-dimensional."
        )

    autocor = fftconvolve(
        cell_mask,
        cell_mask[::-1, ::-1],
        mode="full",
    )

    y0, x0 = np.unravel_index(
        np.argmax(autocor),
        autocor.shape,
    )

    mask_area = cell_mask.sum()
    autocor_area = autocor[y0, x0]

    if not np.isclose(mask_area, autocor_area):
        raise ValueError(
            "Autocorrelation centre does not equal mask area."
        )

    return autocor, x0, y0

In [ ]:
results = {
    1: [],
    2: [],
}

r_values = np.linspace(
    0.1,
    250.0,
    20,
)

area_threshold = 10_000

for dataset_id in range(1, 3):

    n_images_dataset = min(
        n_images[dataset_id],
        len(images[dataset_id]),
        len(coord_map[dataset_id]["min_234"]),
        len(labels[dataset_id]),
    )

    for img_idx in range(n_images_dataset):

        print(
            f"Analysing dataset {dataset_id}, "
            f"image {img_idx}..."
        )

        # Lysosome coordinates
        points = np.asarray(
            coord_map[dataset_id]["min_234"][img_idx],
            dtype=float,
        )

        # Ensure points have shape (n, 2)
        if points.ndim != 2 or points.shape[1] != 2:
            print(
                f"  Skipping: invalid point shape "
                f"{points.shape}"
            )
            continue

        if len(points) < 2:
            print(
                "  Skipping: fewer than two points."
            )
            continue

        # Create one verified 2D cell mask
        cell_mask = create_2d_cell_mask(
            images[dataset_id][img_idx]
        )

        cell_area = int(cell_mask.sum())

        if cell_area < area_threshold:
            print(
                f"  Skipping: cell area={cell_area}, "
                f"threshold={area_threshold}"
            )
            continue

        # Confirm that all points are inside the image bounds
        height, width = cell_mask.shape

        x_coords = np.rint(points[:, 0]).astype(int)
        y_coords = np.rint(points[:, 1]).astype(int)

        in_image = (
            (x_coords >= 0)
            & (x_coords < width)
            & (y_coords >= 0)
            & (y_coords < height)
        )

        if not np.all(in_image):
            print(
                f"  Warning: "
                f"{np.sum(~in_image)} points outside image bounds."
            )

        # Points should normally be inside the cell mask.
        # Do not silently remove them; report them instead.
        valid_coordinate_points = points[in_image]
        valid_x = x_coords[in_image]
        valid_y = y_coords[in_image]

        points_inside_cell = cell_mask[
            valid_y,
            valid_x,
        ].astype(bool)

        n_outside_cell = np.sum(
            ~points_inside_cell
        )

        if n_outside_cell > 0:
            print(
                f"  Warning: {n_outside_cell} points "
                "are outside the cell mask."
            )

        # Construct translation-overlap lookup table
        autocor, x0, y0 = (
            create_mask_autocorrelation(cell_mask)
        )

        # Estimate translation-corrected K
        K = ripley_k_translation(
            points=points,
            r_values=r_values,
            autocor=autocor,
            x0=x0,
            y0=y0,
        )

        label_info = labels[dataset_id][img_idx]

        results[dataset_id].append({
            "dataset_id": dataset_id,
            "img_idx": img_idx,
            "label": label_info[1],
            "K": K,
            "r": r_values.copy(),
            "n_points": len(points),
            "cell_area": cell_area,
            "mask_shape": cell_mask.shape,
            "autocor": autocor,
            "x0": x0,
            "y0": y0,
        })

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ------------------------------------------------------------------
# General style and output directory
# ------------------------------------------------------------------

output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


# ------------------------------------------------------------------
# Flatten Ripley results into long-form data
# ------------------------------------------------------------------

PIXEL_TO_UM = 0.0739859

df_list = []

for dataset_id, entries in results.items():
    for entry in entries:
        r_pixels = np.asarray(entry["r"], dtype=float)
        k_pixels_squared = np.asarray(entry["K"], dtype=float)

        valid = np.isfinite(r_pixels) & np.isfinite(k_pixels_squared)
        valid &= k_pixels_squared >= 0

        # Convert r from pixels to micrometres
        r_um = r_pixels[valid] * PIXEL_TO_UM

        # K is in pixel², so sqrt(K / pi) is in pixels.
        # Convert the resulting L(r) value to micrometres.
        l_um = (
            np.sqrt(k_pixels_squared[valid] / np.pi)
            * PIXEL_TO_UM
            - r_um
        )

        for r_value, l_value in zip(r_um, l_um):
            df_list.append(
                {
                    "r": r_value,
                    "L": l_value,
                    "Condition": str(entry["label"]),
                    "Dataset": f"Dataset {dataset_id}",
                }
            )

df_results = pd.DataFrame(df_list)


# ------------------------------------------------------------------
# Condition order and colours
# ------------------------------------------------------------------

condition_order = [
    "dox-",
    "dox+/PhTau-",
    "dox+/PhTau+",
]

palette = {
    "dox-": "green",
    "dox+/PhTau-": "orange",
    "dox+/PhTau+": "red",
}


# ------------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------------

def add_ripley_plot(data, dataset_name, ax, title=True, show_legend=True):
    """Add one L(r) - r plot with distances in micrometres and horizontal legend."""

    sub_df = data[data["Dataset"] == dataset_name].copy()
    sub_df = sub_df.dropna(subset=["r", "L"])

    active_condition_order = [
        condition
        for condition in condition_order
        if condition in sub_df["Condition"].unique()
    ]

    if not sub_df.empty and active_condition_order:
        sns.lineplot(
            data=sub_df,
            x="r",
            y="L",
            hue="Condition",
            hue_order=active_condition_order,
            palette=palette,
            errorbar=("ci", 95),
            linewidth=2.5,
            legend=show_legend,
            ax=ax,
        )

        # Mark the maximum mean L(r) - r value
        for condition in active_condition_order:
            condition_data = sub_df[
                sub_df["Condition"] == condition
            ]

            mean_curve = (
                condition_data
                .groupby("r", as_index=False)["L"]
                .mean()
            )

            if mean_curve.empty:
                continue

            peak_row = mean_curve.loc[mean_curve["L"].idxmax()]

            ax.plot(
                peak_row["r"],
                peak_row["L"],
                marker="o",
                color=palette[condition],
                markersize=8,
                markeredgecolor="white",
                markeredgewidth=1.5,
                zorder=5,
            )

        # CSR reference line: L(r) - r = 0
        ax.axhline(
            0,
            color="#333333",
            linestyle="--",
            linewidth=1.5,
            zorder=1,
        )

    elif sub_df.empty:
        ax.text(
            0.5,
            0.5,
            "No Data Available",
            ha="center",
            va="center",
            color="gray",
            transform=ax.transAxes,
        )

    else:
        ax.text(
            0.5,
            0.5,
            "No treatment-group data available",
            ha="center",
            va="center",
            color="gray",
            transform=ax.transAxes,
        )

    if title:
        ax.set_title(
            dataset_name,
            fontsize=14,
            fontweight="bold",
        )

    ax.set_xlabel("Neighbourhood radius (µm)", fontsize=12)
    ax.set_ylabel("L(r) - r (µm)", fontsize=12)
    ax.tick_params(labelsize=11)
    ax.set_xlim(left=0)

    if show_legend:
        handles, labels = ax.get_legend_handles_labels()

        handles.append(
            Line2D(
                [0],
                [0],
                color="#333333",
                linestyle="--",
                linewidth=1.5,
            )
        )
        labels.append("baseline (CSR)")

        # Horizontal legend placed above the plot area
        ax.legend(
            handles=handles,
            labels=labels,
            title="Condition",
            fontsize=10,
            title_fontsize=11,
            frameon=True,
            ncol=len(handles),
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
        )


# ------------------------------------------------------------------
# Dataset 1 and Dataset 2 side-by-side
# ------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharex=True,
    sharey=True,
)

add_ripley_plot(df_results, "Dataset 1", axes[0], show_legend=False)
add_ripley_plot(df_results, "Dataset 2", axes[1], show_legend=True)

fig.savefig(
    output_dir / "ripley_L_minus_r_dataset_1_and_2.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------------
# Combined dataset as a separate plot
# ------------------------------------------------------------------

df_combined = df_results.copy()
df_combined["Dataset"] = "Combined"

fig, ax = plt.subplots(figsize=(5, 3))

add_ripley_plot(
    df_combined,
    "Combined",
    ax,
    title=False,
    show_legend=True,
)

fig.savefig(
    output_dir / "ripley_L_minus_r_combined.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

### Analysis by annotation

In [ ]:
CLUSTER_CSV = "Lysosome_annotations_combined.csv"

annotations = []
valid_indices = []

with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    # Use DictReader for easier access to named columns
    reader = csv.DictReader(csvfile)
    
    for i, row in enumerate(reader):
        # 1. Filter out "NoClass" (annotation -1.0)
        if row['annotation_names'] == "NoClass" or float(row['annotations']) == -1.0:
            continue
            
        # 2. Extract relevant data
        ann_name = row['annotation_names']
        phtau_val = row['PhTau_cluster']
        treatment_val = row['treatment']
        
        # 3. Create descriptive label (e.g., "normal, PhTau+")
        phtau_status = "PhTau+" if phtau_val == '1' else "PhTau-"
        full_label = f"{ann_name}, {phtau_status}, {treatment_val}"
        
        # Store label and keep track of the original index to filter images later
        annotations.append([row['annotations'], full_label])
        valid_indices.append(i)

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns


# ------------------------------------------------------------------
# General settings
# ------------------------------------------------------------------

PIXEL_TO_UM = 0.0739859
CLUSTER_CSV = "Lysosome_annotations_combined.csv"

output_dir = Path("Figures")
output_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(
    style="whitegrid",
    rc={
        "grid.linewidth": 0.5,
        "grid.color": "#E5E5E5",
    },
)


# ------------------------------------------------------------------
# Annotation order and colours
# ------------------------------------------------------------------

annotation_order = [
    "normal",
    "enlarged",
    "dispersed",
    "enlarged&dispersed",
    "big clusters",
    "other/atypical",
]

annotation_colours = {
    "big clusters": "#F1C40F",
    "dispersed": "#3498DB",
    "enlarged": "#E74C3C",
    "enlarged&dispersed": "#9B59B6",
    "normal": "#2ECC71",
    "other/atypical": "#95A5A6",
}


# ------------------------------------------------------------------
# Read annotations and retain the original image indices
# ------------------------------------------------------------------

annotations = []
valid_indices = []

with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)

    for image_idx, row in enumerate(reader):
        annotation_name = row["annotation_names"].strip()
        annotation_value = float(row["annotations"])

        if annotation_name == "NoClass" or annotation_value == -1.0:
            continue

        # Handles labels such as:
        # "normal, PhTau-, dox-"
        # as well as already-clean labels such as:
        # "normal"
        clean_annotation = annotation_name.split(",")[0].strip()

        annotations.append(clean_annotation)
        valid_indices.append(image_idx)


# ------------------------------------------------------------------
# Retrieve Dataset 1 Ripley results
# ------------------------------------------------------------------

dataset_1_entries = results.get(1, results.get("Dataset 1", []))

if not dataset_1_entries:
    raise ValueError("No Dataset 1 Ripley results were found.")


# Use stored image indices when available.
# This is safer if some images were skipped during Ripley analysis.
entries_by_img_idx = {
    entry["img_idx"]: entry
    for entry in dataset_1_entries
    if "img_idx" in entry
}

df_list = []

for annotation_label, image_idx, positional_idx in zip(
    annotations,
    valid_indices,
    range(len(annotations)),
):
    if entries_by_img_idx:
        entry = entries_by_img_idx.get(image_idx)

        if entry is None:
            continue
    else:
        # Fallback for older results without stored img_idx
        if positional_idx >= len(dataset_1_entries):
            continue

        entry = dataset_1_entries[positional_idx]

    r_pixels = np.asarray(entry["r"], dtype=float)
    k_pixels_squared = np.asarray(entry["K"], dtype=float)

    valid = np.isfinite(r_pixels) & np.isfinite(k_pixels_squared)
    valid &= k_pixels_squared >= 0

    if not np.any(valid):
        continue

    # Convert r from pixels to micrometres
    r_um = r_pixels[valid] * PIXEL_TO_UM

    # K is in pixel².
    # Therefore sqrt(K / pi) is in pixels and must also be converted.
    l_um_minus_r_um = (
        np.sqrt(k_pixels_squared[valid] / np.pi) * PIXEL_TO_UM
        - r_um
    )

    for r_value, l_value in zip(r_um, l_um_minus_r_um):
        df_list.append(
            {
                "r": r_value,
                "L": l_value,
                "Annotation": annotation_label,
            }
        )

df_results = pd.DataFrame(df_list)


# ------------------------------------------------------------------
# Retain only annotation categories present in Dataset 1
# ------------------------------------------------------------------

if not df_results.empty:
    active_annotation_order = [
        annotation
        for annotation in annotation_order
        if annotation in df_results["Annotation"].unique()
    ]
else:
    active_annotation_order = []


# ------------------------------------------------------------------
# Generate the annotation-based Ripley plot
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 3))

if not df_results.empty and active_annotation_order:

    sns.lineplot(
        data=df_results,
        x="r",
        y="L",
        hue="Annotation",
        hue_order=active_annotation_order,
        palette=annotation_colours,
        errorbar=("ci", 95),
        linewidth=2.5,
        ax=ax,
        legend=True,
    )

    # Mark the maximum mean L(r) - r value for each annotation
    for annotation in active_annotation_order:
        annotation_data = df_results[
            df_results["Annotation"] == annotation
        ]

        mean_curve = (
            annotation_data
            .groupby("r", as_index=False)["L"]
            .mean()
        )

        if mean_curve.empty:
            continue

        peak_row = mean_curve.loc[mean_curve["L"].idxmax()]

        ax.plot(
            peak_row["r"],
            peak_row["L"],
            marker="o",
            color=annotation_colours[annotation],
            markersize=8,
            markeredgecolor="white",
            markeredgewidth=1.5,
            zorder=5,
        )

    # Complete spatial randomness reference
    ax.axhline(
        0,
        color="#333333",
        linestyle="--",
        linewidth=1.5,
        zorder=1,
    )

    # Add the CSR baseline to the Seaborn legend
    handles, labels = ax.get_legend_handles_labels()

    handles.append(
        Line2D(
            [0],
            [0],
            color="#333333",
            linestyle="--",
            linewidth=1.5,
        )
    )
    labels.append("baseline (CSR)")

    ax.legend(
        handles=handles,
        labels=labels,
        title="Annotation",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=11,
        title_fontsize=12,
        frameon=True,
        facecolor="white",
        edgecolor="#E5E5E5",
    )

else:
    ax.text(
        0.5,
        0.5,
        "No Data Available",
        ha="center",
        va="center",
        color="gray",
        transform=ax.transAxes,
    )


# ------------------------------------------------------------------
# Formatting
# ------------------------------------------------------------------

# ax.set_title(
#     "Lysosome Spatial Organisation by Annotation",
#     fontsize=14,
#     fontweight="bold",
#     pad=12,
# )

ax.set_xlabel(
    "Neighbourhood radius (µm)",
    fontsize=12,
    labelpad=8,
)

ax.set_ylabel(
    "L(r) - r",
    fontsize=12,
    labelpad=8,
)

ax.tick_params(labelsize=11)
ax.set_xlim(left=0)

fig.savefig(
    output_dir / "ripley_L_minus_r_by_annotation.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)